<a href="https://colab.research.google.com/github/eren-darici/oil-blending-lp/blob/main/Project_Draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [292]:
!pip install pulp
import pulp as pl
import pandas as pd
import numpy as np

In [293]:
# Read data
# DATA_PATH = "data.xlsx"
DATA_PATH = "https://github.com/eren-darici/oil-blending-lp/blob/main/data.xlsx?raw=true"
master_data = pd.read_excel(DATA_PATH, sheet_name="master_data")
indexed_data = pd.read_excel(DATA_PATH, sheet_name="indexed_data")

price_data = pd.read_excel(DATA_PATH, sheet_name="price_data")
indexed_price_data = pd.read_excel(DATA_PATH, sheet_name="indexed_price_data")

In [294]:
# Create oils list to create decision variables
crude_oils = list(set(master_data['Oil']))

# Create the decision variables
var_oils = pl.LpVariable.dicts("oils", crude_oils, lowBound=0)
print(var_oils)

{'A.L': oils_A.L, 'M': oils_M, 'L.G': oils_L.G, 'Gu': oils_Gu, 'K': oils_K, 'W.D': oils_W.D, 'Cond.': oils_Cond., 'L': oils_L, 'Q': oils_Q, 'B': oils_B}


In [295]:
crude_oils

['A.L', 'M', 'L.G', 'Gu', 'K', 'W.D', 'Cond.', 'L', 'Q', 'B']

In [296]:
# LPG yields
# LPG yield from disttillation is x11, LPG yield from bed reformer x12, total x13 in the paper
# LPG yield decision variables
lpg_yield_from = ["distillation", "bed_reformer", "total"]
var_lpg_yields = pl.LpVariable.dicts("lpg", lpg_yield_from, lowBound=0)

In [297]:
# Product yields
# naphtha x14, kerosene x15, gas oil x16, fuel oil x17
products = ["naphtha", "kerosene", "gas oil", "fuel oil"]
var_product_yields = pl.LpVariable.dicts("product", products, lowBound=0)

In [298]:
# imported
# Naptha x18 , total naptha x19
# Reformer capacity x20, yield of reformate x21 (gasoline92)
# remaining available naptha after feeding the reformer x22
# gasoline 95 is x23
# gasoline80 is x24
otherx = ["imported_naptha", "total_naptha", "reformer_capacity", "gasoline92", "remaining_naptha", "gasoline95", "gasoline80"]
var_otherx = pl.LpVariable.dicts("otherx", otherx, lowBound=0)

In [299]:
var_product_yields

{'naphtha': product_naphtha,
 'kerosene': product_kerosene,
 'gas oil': product_gas_oil,
 'fuel oil': product_fuel_oil}

In [300]:
# The revenue = the sum of products yield multiplied by the price of each product
revenue = 0
np.array(indexed_data["Li"])

array([0.5, 1.2, 0.9, 2.1, 1.3, 1.3, 1.8, 1.1, 3. , 0.8])

In [301]:
indexed_data

,Oil,Pi,Ti,Ci,gi,Si,SUi,Wi,Ai,CCRi,...,SUFi,WFi,ASFi,Vni,Ni,CTi,CUi,API,F,VI
0,L,58.858,3700,6.80,0.925,42.0,2.85,3.72,13.0,11.80,...,3.77,3.77,14.7,152,131.00,0.5,2,21.472973,6.799252,1.569112
1,M,61.865,3800,7.16,0.878,33.0,2.00,2.92,5.7,6.24,...,3.10,3.90,4.9,100,73.00,0.5,2,29.661731,7.163221,1.274789
2,Gu,60.914,11300,7.22,0.871,28.0,1.61,3.80,7.0,5.20,...,2.40,4.80,2.9,76,40.00,0.5,2,30.956946,7.220790,1.231835
3,W.D,62.500,3200,7.75,0.812,18.0,0.30,6.10,11.0,1.52,...,0.80,8.50,1.6,20,11.00,1.5,2,42.761084,7.745453,1.015234
4,A.L,58.589,13300,7.30,0.862,15.0,1.85,3.50,5.0,4.00,...,2.50,4.50,1.5,43,13.75,0.5,2,32.653132,7.296181,1.076318
5,K,63.416,7100,7.22,0.872,38.0,2.80,1.80,14.0,5.80,...,3.45,2.30,2.5,62,22.00,0.5,2,30.770642,7.212509,1.310677
6,Q,63.754,3700,7.29,0.862,20.0,1.10,6.30,16.0,4.04,...,3.40,7.40,1.9,11,4.00,0.8,2,32.653132,7.296181,1.250025
7,B,59.795,6300,7.19,0.875,14.0,2.60,3.90,11.2,6.33,...,3.20,5.30,5.0,66,19.00,0.5,2,30.214286,7.187781,1.222159
8,Cond.,58.752,1300,8.20,0.767,9.0,0.50,6.20,8.0,0.06,...,0.20,7.80,0.5,9,6.00,0.5,2,52.985007,8.199880,0.776915
9,L.G,58.934,2600,7.12,0.883,12.8,1.63,2.20,12.0,5.60,...,2.57,4.20,5.1,100,75.00,0.5,2,28.749151,7.122659,1.110196


In [302]:
# Maps - dicts
li_map = (indexed_data.set_index("Oil")["Li"] / 100).to_dict()
nai_map = (indexed_data.set_index("Oil")["NAi"] / 100).to_dict()
ki_map = (indexed_data.set_index("Oil")["Ki"] / 100).to_dict()
gi_map = (indexed_data.set_index("Oil")["Gi"] / 100).to_dict()
fi_map = (indexed_data.set_index("Oil")["Fi"] / 100).to_dict()

pi_map = indexed_data.set_index("Oil")["Pi"].to_dict()
cti_map = indexed_data.set_index("Oil")["CTi"].to_dict()
cui_map = indexed_data.set_index("Oil")["CUi"].to_dict()

ci_map = indexed_data.set_index("Oil")["Ci"].to_dict()
gravityi_map = indexed_data.set_index("Oil")["gi"].to_dict()
si_map = (indexed_data.set_index("Oil")["Si"] / 100).to_dict()
sui_map = (indexed_data.set_index("Oil")["SUi"] / 100).to_dict()
ai_map = indexed_data.set_index("Oil")["Ai"].to_dict()
wi_map = (indexed_data.set_index("Oil")["Wi"] / 100).to_dict()

vi_map = indexed_data.set_index("Oil")["VI"].to_dict()

asi_map = (indexed_data.set_index("Oil")['SARA_ASi'] / 100).to_dict()
ri_map = (indexed_data.set_index("Oil")['SARA_Ri'] / 100).to_dict()
ari_map = (indexed_data.set_index("Oil")['SARA_ARi'] / 100).to_dict()
saturates_map = (indexed_data.set_index("Oil")['SARA_Pi'] / 100).to_dict()
sufi_map = (indexed_data.set_index("Oil")['SUFi'] / 100).to_dict()
wfi_map = (indexed_data.set_index("Oil")['WFi'] / 100).to_dict()
asfi_map = (indexed_data.set_index("Oil")['ASFi'] / 100).to_dict()
vni_map = (indexed_data.set_index("Oil")['Vni'] / 100).to_dict()
ni_map = (indexed_data.set_index("Oil")['Ni'] / 100).to_dict()

ti_map = indexed_data.set_index("Oil")["Ti"].to_dict()

fBBL_map = indexed_data.set_index("Oil")["F"].to_dict()

revenue_map = indexed_price_data.set_index("Yield")['Price'].to_dict()


comp_map = {
    "Li": li_map,
    "NAi": nai_map,
    "Ki": ki_map,
    "Gi": gi_map,
    "Fi": fi_map
}

In [304]:
# Revenue function (revenue_i * li * xi)
revenue_components = ["Li", "NAi", "Ki", "Gi", "Fi"]

revenue = pl.lpSum(
    revenue_map[c] * globals()[f"{c.lower()}_map"][o] * var_oils[o]
    for c in revenue_components
    for o in crude_oils
) + 0.03 * revenue_map["Li"] * var_otherx["reformer_capacity"] + 0.97 * revenue_map["gasoline92"] * var_otherx["reformer_capacity"] + revenue_map["gasoline80"] * var_otherx["gasoline80"]


In [305]:
# Cost Function (conversion only applied for Pi (in bbls))
cost_components = ["Pi", "CTi", "CUi"]

cost = pl.lpSum(
    globals()[f"{c.lower()}_map"][o] * var_oils[o] *
    (fBBL_map[o] if c == "Pi" else 1)
    for c in cost_components
    for o in crude_oils
) + revenue_map["NAi"] * var_otherx["reformer_capacity"] + 1 *var_otherx["reformer_capacity"] + revenue_map["gasoline95"] * var_otherx["gasoline95"] + revenue_map["NAi"] * var_otherx["remaining_naptha"]

In [306]:
# Create the model
model = pl.LpProblem("OilBlending", pl.LpMaximize)

In [307]:
# Objective function
model += (revenue - cost)

In [308]:
# Operating Capacity Constraint (17)
model += pl.lpSum(ci_map[o] * var_oils[o] for o in crude_oils) <= 150000

In [309]:
# Specific Gravity Constraint (18)
model += pl.lpSum(ci_map[o] * gravityi_map[o] * var_oils[o] for o in crude_oils) <= 0.867 * pl.lpSum(ci_map[o] * var_oils[o] for o in crude_oils)

In [310]:
# Fuel Oil Production Constraint (19) #PROBLEM????????????
model += pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils) >= 10600

In [311]:
# Salt Content Constraint (20)
model += pl.lpSum(si_map[o] * var_oils[o] for o in crude_oils) <= 30 * pl.lpSum(var_oils[o] for o in crude_oils)

In [312]:
# Sulfur Content Constraint (21)
model += pl.lpSum(sui_map[o] * var_oils[o] for o in crude_oils) <= 2.1 * pl.lpSum(var_oils[o] for o in crude_oils)

In [313]:
# Acidity Number Constraint (22)
model += pl.lpSum(ai_map[o] * var_oils[o] for o in crude_oils) <= 12.1 * pl.lpSum(var_oils[o] for o in crude_oils)

In [314]:
# Wax Number Constraint (23)
model += pl.lpSum(wi_map[o] * var_oils[o] for o in crude_oils) <= 3.8 * pl.lpSum(var_oils[o] for o in crude_oils)

In [315]:
# Viscosity Index Constraint (25)
model += pl.lpSum(vi_map[o] * var_oils[o] for o in crude_oils) <= 1.232 * pl.lpSum(var_oils[o] for o in crude_oils)

In [316]:
# Compatability Constraint (26)
model += pl.lpSum(asi_map[o] * var_oils[o] for o in crude_oils) <= 0.35 * pl.lpSum(ri_map[o] * var_oils[o] for o in crude_oils)

In [317]:
# Collodial Instability Index Constraint (27) ??????
# model += pl.lpSum((pi_map[o] + asi_map[o]) * var_oils[o] for o in crude_oils) <= 0.7 * pl.lpSum((ari_map[o] + ri_map[o]) * var_oils[o] for o in crude_oils)

In [318]:
# Crude Oil Availability Constraint (28-37)
for oil in crude_oils:
    model += var_oils[oil] <= ti_map[oil]

In [319]:
# Sulfur Content in Fuel Oil Constraint (38)
model += pl.lpSum(sufi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 3 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils)

In [320]:
# Wax Content in Fuel Oil Constraint (39)
model += pl.lpSum(wfi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 7 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils)

In [321]:
# Asphaltane Content in Fuel Oil Constraint (40)
model += pl.lpSum(asfi_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 5 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils)

In [322]:
# Vanadium Content in Fuel Oil Constraint (41)
model += pl.lpSum(vni_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 70 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils)

In [323]:
# Nickel Content in Fuel Oil Constraint (42)
model += pl.lpSum(ni_map[o] * fi_map[o] * var_oils[o] for o in crude_oils) <= 40 * pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils)

In [324]:
# LPG Yield from Distillation (43)
model += var_lpg_yields["distillation"] == pl.lpSum(li_map[o] * var_oils[o] for o in crude_oils)

In [325]:
# LPG Yield from Reformer (44)
model += var_lpg_yields["bed_reformer"] - 0.03 * var_otherx["reformer_capacity"] == 0

In [326]:
# Total LPG Yield (45)
model += var_lpg_yields["total"] - var_lpg_yields["distillation"] - var_lpg_yields["distillation"] == 0

In [327]:
# Naphtha Yield from Distillation (46)
model += var_product_yields["naphtha"] == pl.lpSum(nai_map[o] * var_oils[o] for o in crude_oils)

In [328]:
# Kerosene Yield from Distillation (47)
model += var_product_yields["kerosene"] == pl.lpSum(ki_map[o] * var_oils[o] for o in crude_oils)

In [329]:
# Gas Oil Yield from Distillation (48)
model += var_product_yields["gas oil"] == pl.lpSum(gi_map[o] * var_oils[o] for o in crude_oils)

In [330]:
# Fuel Oil Yield from Distillation (49)
model += var_product_yields["fuel oil"] == pl.lpSum(fi_map[o] * var_oils[o] for o in crude_oils)

In [331]:
# Total Naphtha Available (50)
model += var_otherx["total_naptha"] - var_product_yields['naphtha'] - var_otherx["imported_naptha"] == 0

In [332]:
# Imported Naphtha Constraint (51)
model += var_otherx["imported_naptha"] <= 0

In [333]:
# Naphtha Distribution Constraint (52)
model += var_otherx["total_naptha"] - var_otherx["reformer_capacity"] - var_otherx["remaining_naptha"] == 0

In [334]:
# Fixed Bed Reformer Capacity Constraint (53)
model += var_otherx["reformer_capacity"] <= 50000

In [335]:
# Reformate Yield from Reformer (54)
model += 1.031 * var_otherx["gasoline92"] - var_otherx["reformer_capacity"] == 0

In [336]:
# Gasoline 80 Blending (55)
model += var_otherx["gasoline95"] * 0.4 == 0.6 * var_otherx["remaining_naptha"]

In [337]:
# Gasoline 80 Yield (56)
model += var_otherx["gasoline80"] - var_otherx['remaining_naptha'] - var_otherx['gasoline95'] == 0

In [338]:
# Gasoline 80-92 Demands (57-58)

model += var_otherx["gasoline80"] >= 4000
model += var_otherx["gasoline92"] >= 4500

In [339]:
# Imported Gasoline 95 Constraint (59)
model += var_otherx["gasoline95"] <= 10000

In [340]:
model.solve()

-1

In [341]:
pl.LpStatus[model.solve()]

'Infeasible'

In [342]:
pl.value(model.objective)

-4989870.387881339

# Utilities for Printing Purposes

In [346]:
model_map = {
    # Crude oils (X1–X10)
    "X1 (L)": var_oils["L"],
    "X2 (M)": var_oils["M"],
    "X3 (Gu)": var_oils["Gu"],
    "X4 (W.D)": var_oils["W.D"],
    "X5 (A.L)": var_oils["A.L"],
    "X6 (K)": var_oils["K"],
    "X7 (Q)": var_oils["Q"],
    "X8 (B)": var_oils["B"],
    "X9 (Cond.)": var_oils["Cond."],
    "X10 (L.G)": var_oils["L.G"],

    # LPG yields (X11–X13)
    "X11 (LPG distillation)": var_lpg_yields["distillation"],
    "X12 (LPG reformer)": var_lpg_yields["bed_reformer"],
    "X13 (Total LPG)": var_lpg_yields["total"],

    # Other variables (X18–X23 as available in your model)
    "X18 (Imported naphtha)": var_otherx["imported_naptha"],
    "X19 (Total naphtha)": var_otherx["total_naptha"],

    "X20 (Reformer capacity)": var_otherx["reformer_capacity"],
    "X21 (Gasoline 92)": var_otherx["gasoline92"],
    "X22 (Remaining naphtha)": var_otherx["remaining_naptha"],
    "X23 (Gasoline 95)": var_otherx["gasoline95"],
}

In [347]:
for k, v in model_map.items():
    print(k, v.value())

X1 (L) 0.0
X2 (M) 0.0
X3 (Gu) -374863.27
X4 (W.D) 3200.0
X5 (A.L) 0.0
X6 (K) 7100.0
X7 (Q) 379943.86
X8 (B) 0.0
X9 (Cond.) 1300.0
X10 (L.G) 0.0
X11 (LPG distillation) 3663.7202
X12 (LPG reformer) 139.185
X13 (Total LPG) 7327.4404
X18 (Imported naphtha) 0.0
X19 (Total naphtha) 6239.5
X20 (Reformer capacity) 4639.5
X21 (Gasoline 92) 4500.0
X22 (Remaining naphtha) 1600.0
X23 (Gasoline 95) 2400.0
